In [1]:
import pandas as pd
from pathlib import Path
from neo4j import GraphDatabase
from getpass import getpass

In [2]:
project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

output_folder = project_folder / "data" / "outputs"

traffic_df = pd.read_csv(output_folder / "traffic_data.csv")
roads_df = pd.read_csv(output_folder / "roads.csv", dtype={"id": str})

traffic_df["roadId"] = traffic_df["roadId"].astype(str)

print("Traffic records:", len(traffic_df))
print("Roads:", len(roads_df))

Traffic records: 864
Roads: 794


In [4]:
data = traffic_df.merge(
    roads_df[["id", "speedLimit"]],
    left_on="roadId",
    right_on="id",
    how="left"
)

print("Combined records:", len(data))
print("Missing speed limits:", data["speedLimit"].isna().sum())

data.head()

Combined records: 864
Missing speed limits: 0


,begin,end,roadId,avgSpeed,density,vehicleCount,id,speedLimit
0,0.0,60.0,-170201513#0,37.260,1.77,1,-170201513#0,50.004
1,0.0,60.0,-170201513#1,30.708,1.95,0,-170201513#1,50.004
2,0.0,60.0,-170204655#0,39.132,1.64,1,-170204655#0,50.004
3,0.0,60.0,-170204655#2,34.776,2.06,1,-170204655#2,50.004
4,0.0,60.0,-170205509#0,50.904,0.15,1,-170205509#0,50.004


In [5]:
data["avgSpeed"] = pd.to_numeric(data["avgSpeed"], errors="coerce")
data["vehicleCount"] = pd.to_numeric(data["vehicleCount"], errors="coerce")

congested = data[
    (data["vehicleCount"] > 0) &
    (data["speedLimit"] > 0) &
    (data["avgSpeed"] >= 0) &
    (data["avgSpeed"] < 0.5 * data["speedLimit"])
].copy()

print("Congestion records detected:", len(congested))

congested[["roadId", "avgSpeed", "speedLimit", "density"]].head()

Congestion records detected: 75


,roadId,avgSpeed,speedLimit,density
12,-28851399#4,39.708,79.992,1.80
47,28851329#0,20.628,50.004,3.45
48,28851342,39.816,79.992,2.82
58,28851494#1,24.264,50.004,0.29
68,449397140,33.948,79.992,5.94


In [6]:
congested = congested.reset_index(drop=True)

congested["congestionId"] = [
    f"SUMO_Congestion_{i+1}"
    for i in range(len(congested))
]

congested["neo4jRoadId"] = "SUMO_" + congested["roadId"]

congested["congestionLevel"] = "High"
congested["source"] = "SUMO_Simulation"

congestion_df = congested[[
    "congestionId",
    "neo4jRoadId",
    "congestionLevel",
    "avgSpeed",
    "density",
    "vehicleCount",
    "begin",
    "end",
    "source"
]]

print("Congestion events:", len(congestion_df))

congestion_df.head()

Congestion events: 75


,congestionId,neo4jRoadId,congestionLevel,avgSpeed,density,vehicleCount,begin,end,source
0,SUMO_Congestion_1,SUMO_-28851399#4,High,39.708,1.80,1,0.0,60.0,SUMO_Simulation
1,SUMO_Congestion_2,SUMO_28851329#0,High,20.628,3.45,1,0.0,60.0,SUMO_Simulation
2,SUMO_Congestion_3,SUMO_28851342,High,39.816,2.82,2,0.0,60.0,SUMO_Simulation
3,SUMO_Congestion_4,SUMO_28851494#1,High,24.264,0.29,1,0.0,60.0,SUMO_Simulation
4,SUMO_Congestion_5,SUMO_449397140,High,33.948,5.94,2,0.0,60.0,SUMO_Simulation


In [7]:
output = output_folder / "congestion.csv"

congestion_df.to_csv(output, index=False)

print("Congestion dataset saved!")
print("Total events:", len(congestion_df))

Congestion dataset saved!
Total events: 75


In [8]:
uri = "bolt://localhost:7687"
username = "neo4j"

password = getpass("Enter Neo4j password: ")

driver = GraphDatabase.driver(
    uri,
    auth=(username, password)
)

driver.verify_connectivity()

print("Connected to Neo4j!")

Connected to Neo4j!


In [9]:
congestion_data = congestion_df.to_dict("records")

query = """
UNWIND $batch AS row

MERGE (c:Congestion {id: row.congestionId})

SET c.congestionLevel = row.congestionLevel,
    c.avgSpeed = row.avgSpeed,
    c.density = row.density,
    c.vehicleCount = row.vehicleCount,
    c.begin = row.begin,
    c.end = row.end,
    c.source = row.source
"""

with driver.session(database="neo4j") as session:

    for i in range(0, len(congestion_data), 500):
        batch = congestion_data[i:i+500]
        session.run(query, batch=batch).consume()

print("Congestion nodes imported!")

Congestion nodes imported!


In [10]:
query = """
UNWIND $batch AS row

MATCH (c:Congestion {id: row.congestionId})
MATCH (r:Road {id: row.neo4jRoadId})

MERGE (c)-[:LOCATED_AT]->(r)
"""

with driver.session(database="neo4j") as session:

    for i in range(0, len(congestion_data), 500):
        batch = congestion_data[i:i+500]
        session.run(query, batch=batch).consume()

print("Congestion-road relationships created!")

Congestion-road relationships created!


In [11]:
driver.close()

print("Neo4j connection closed!")

Neo4j connection closed!
